# **Retrieval**

In LangChain, retrieval refers to the process of accessing and fetching relevant pieces of information from a collection of documents or data sources based on a query. This involves using techniques like similarity search and embedding-based methods to find and return the most pertinent documents or text segments that match the query's context and content.

## Setup

Load and/or check for needed environmental variables

In [ ]:
%%capture
# update or install the necessary libraries
%pip install --upgrade langchain langchain_community langchain_ollama pypdf tiktoken langchain-chroma lark langchain_text_splitters
%pip install --upgrade python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Ollama server URL - override with OLLAMA_BASE_URL env var
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")

# **Vectorstore retrieval**

Vectorstore retrieval is a technique that involves storing and retrieving data using vector representations of text. It enables efficient similarity searches by converting text into high-dimensional vectors and then finding the closest vectors in the database. This method is particularly useful for applications requiring fast and accurate information retrieval based on semantic similarity.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

# Load PDF
loaders = [
    # Duplicate documents on purpose - messy data
    PyPDFLoader("./content/MachineLearning-Lecture01.pdf"),
    PyPDFLoader("./content/MachineLearning-Lecture02.pdf"),
    PyPDFLoader("./content/MachineLearning-Lecture03.pdf")

]
docs = []
for loader in loaders:
    docs.extend(loader.load())

In [ ]:
# Split
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1500,
    chunk_overlap = 150
)

splits = text_splitter.split_documents(docs)

len(splits)

In [ ]:
from langchain_chroma import Chroma

In [ ]:
from langchain_ollama import OllamaEmbeddings

embedding = OllamaEmbeddings(
    model="embeddinggemma:latest",
    base_url=OLLAMA_BASE_URL,
)

# In-memory Chroma stores in one kernel share collections by name, so each store gets its own name
vectordb = Chroma(collection_name="lectures", embedding_function=embedding)
vectordb.reset_collection()  # start empty, so re-running this cell doesn't add duplicate chunks
vectordb.add_documents(splits)

In [ ]:
texts = [
    "The cat is a small, furry animal that likes to sleep.",
    "Some cats are all white and have large bodies.",
    "Cats, also called felines, are popular pets."
]

In [ ]:
smalldb = Chroma(collection_name="cats", embedding_function=embedding)
smalldb.reset_collection()
smalldb.add_texts(texts)

In [ ]:
question = "Tell me about all-white cats with large bodies"

# **Similarity Search**

Similarity search is a method used to find items in a database that are most similar to a given query item based on certain criteria. It involves comparing vector representations of items and retrieving those with the smallest distance or highest similarity score. This approach is commonly used in applications like document retrieval, image matching, and recommendation systems.

In [ ]:
similarity_retriever = smalldb.as_retriever(search_kwargs={"k": 2})
similarity_retriever.invoke(question)

In [ ]:
mmr_retriever = smalldb.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 2, "fetch_k": 3},
)
mmr_retriever.invoke(question)

## Addressing Diversity: Maximum marginal relevance

Last class we introduced one problem: how to enforce diversity in the search results.

`Maximum marginal relevance` strives to achieve both relevance to the query *and diversity* among the results.

Maximum Marginal Relevance (MMR) is a technique used in information retrieval to enhance the diversity of search results. It aims to select documents that are not only relevant to a query but also diverse from each other. This diversity helps provide a broader perspective or coverage of the topic being queried.

In [ ]:
question = "what did they say about matlab?"
similarity_retriever = vectordb.as_retriever(search_kwargs={"k": 3})
docs_ss = similarity_retriever.invoke(question)

In [ ]:
docs_ss[0].page_content[:100]

In [ ]:
docs_ss[1].page_content[:100]

Note the difference in results with `MMR`.

In [ ]:
mmr_retriever = vectordb.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3},
)
docs_mmr = mmr_retriever.invoke(question)

In [ ]:
docs_mmr[0].page_content[:100]

In [ ]:
docs_mmr[1].page_content[:100]

## **Addressing Specificity: working with metadata**

In last lecture, we showed that a question about the third lecture can include results from other lectures as well.

To address this, many vectorstores support operations on `metadata`.

`metadata` provides context for each embedded chunk.

Metadata filtering refers to the process of selectively retrieving or excluding data based on predefined metadata attributes or criteria. In the context of information retrieval systems like LangChain, metadata filtering allows users to narrow down search results by specifying metadata tags or attributes associated with documents or data chunks.

In [ ]:
question = "what did they say about regression in the third lecture?"

In [ ]:
filtered_retriever = vectordb.as_retriever(
    search_kwargs={
        "k": 3,
        "filter": {"source": "./content/MachineLearning-Lecture03.pdf"},
    }
)
docs = filtered_retriever.invoke(question)

In [ ]:
for d in docs:
    d.metadata

# **Let's Do an Activity**

## **Objective**

Explore different retrieval techniques in LangChain to fetch relevant documents based on queries.

## **Scenario**

You are building a document retrieval system for a research library. This activity will help you understand and implement various retrieval methods to efficiently fetch documents related to specific topics or queries.

## **Steps**

* Load Documents
* Embedding and Vector Store
* Similarity Search
* Maximum Marginal Relevance (MMR)
* Metadata Filtering
* Alternative Retrieval Methods